## Setup

In [11]:
import sys
from pathlib import Path

# Get the BICEP root directory (three levels up from this notebook)
bicep_root = Path.cwd().parent.parent.parent
if str(bicep_root) not in sys.path:
    sys.path.insert(0, str(bicep_root))

import pandas as pd
import numpy as np
from scipy import stats

print("Environment setup complete!")

Environment setup complete!


## Why Use Distributions?

Infrastructure costs vary based on:

1. **Building characteristics** - Size, age, existing equipment
2. **Regional variations** - Labor costs, material prices
3. **Technical factors** - Complexity of upgrades, site conditions
4. **Market factors** - Supply chain, contractor availability

Instead of using a single cost estimate, BICEP uses probability distributions to capture this uncertainty.

## Built-in Distributions in BICEP

### 1. Panel Utilization Distribution

Represents how fully existing electrical panels are utilized.

In [28]:
# Load results to analyze cost distributions
data_path = bicep_root / 'data' / 'parsed_inputs'

print("Loading High scenario results...")
results = pd.read_csv(data_path / 'bicep_results_high_all_states.csv')

print(f"Loaded {len(results):,} buildings\n")

# Analyze the cost distribution across buildings
print("Cost Distribution Statistics (All Buildings):")
print(f"  Mean: ${results['weighted_cost'].mean():,.0f}")
print(f"  Median: ${results['weighted_cost'].median():,.0f}")
print(f"  Std Dev: ${results['weighted_cost'].std():,.0f}")
print(f"  Min: ${results['weighted_cost'].min():,.0f}")
print(f"  Max: ${results['weighted_cost'].max():,.0f}")
print(f"  25th percentile: ${results['weighted_cost'].quantile(0.25):,.0f}")
print(f"  75th percentile: ${results['weighted_cost'].quantile(0.75):,.0f}")

Loading High scenario results...
Loaded 884,596 buildings

Cost Distribution Statistics (All Buildings):
  Mean: $43,821
  Median: $24,486
  Std Dev: $768,545
  Min: $-172,948,434
  Max: $507,416
  25th percentile: $9,787
  75th percentile: $60,247


### 2. PV Sizing Distribution

Represents the relationship between building loads and solar PV system size.

In [29]:
# Analyze cost distribution by residential vs commercial
print("\n" + "="*70)
print("COST DISTRIBUTION BY BUILDING TYPE")
print("="*70)

residential = results[results['residential'] == 1]
commercial = results[results['residential'] == 0]

print(f"\nResidential ({len(residential):,} buildings):")
print(f"  Mean cost: ${residential['weighted_cost'].mean():,.0f}")
print(f"  Median cost: ${residential['weighted_cost'].median():,.0f}")
print(f"  Std Dev: ${residential['weighted_cost'].std():,.0f}")
print(f"  5th percentile: ${residential['weighted_cost'].quantile(0.05):,.0f}")
print(f"  95th percentile: ${residential['weighted_cost'].quantile(0.95):,.0f}")

print(f"\nCommercial ({len(commercial):,} buildings):")
print(f"  Mean cost: ${commercial['weighted_cost'].mean():,.0f}")
print(f"  Median cost: ${commercial['weighted_cost'].median():,.0f}")
print(f"  Std Dev: ${commercial['weighted_cost'].std():,.0f}")
print(f"  5th percentile: ${commercial['weighted_cost'].quantile(0.05):,.0f}")
print(f"  95th percentile: ${commercial['weighted_cost'].quantile(0.95):,.0f}")


COST DISTRIBUTION BY BUILDING TYPE

Residential (548,916 buildings):
  Mean cost: $58,230
  Median cost: $37,504
  Std Dev: $819,177
  5th percentile: $5,242
  95th percentile: $230,414

Commercial (335,680 buildings):
  Mean cost: $10,239
  Median cost: $9,082
  Std Dev: $633,801
  5th percentile: $2,332
  95th percentile: $51,969


### 3. Panel Upgrade Cost Distribution

Represents the variation in electrical panel upgrade costs based on building type and regional factors.

In [30]:
# Analyze how costs vary within building types
print("\n" + "="*70)
print("COST VARIATION WITHIN BUILDING TYPES")
print("="*70)

# For residential buildings
res_with_costs = residential[residential['weighted_cost'] > 0]
res_without_costs = residential[residential['weighted_cost'] == 0]

print(f"\nResidential Buildings:")
print(f"  With upgrades: {len(res_with_costs):,} ({len(res_with_costs)/len(residential)*100:.1f}%)")
print(f"  Without upgrades: {len(res_without_costs):,} ({len(res_without_costs)/len(residential)*100:.1f}%)")

if len(res_with_costs) > 0:
    print(f"\n  Buildings with upgrades - Cost distribution:")
    print(f"    Mean: ${res_with_costs['weighted_cost'].mean():,.0f}")
    print(f"    Median: ${res_with_costs['weighted_cost'].median():,.0f}")
    print(f"    Range: ${res_with_costs['weighted_cost'].min():,.0f} - ${res_with_costs['weighted_cost'].max():,.0f}")

# For commercial buildings
com_with_costs = commercial[commercial['weighted_cost'] > 0]
com_without_costs = commercial[commercial['weighted_cost'] == 0]

print(f"\nCommercial Buildings:")
print(f"  With upgrades: {len(com_with_costs):,} ({len(com_with_costs)/len(commercial)*100:.1f}%)")
print(f"  Without upgrades: {len(com_without_costs):,} ({len(com_without_costs)/len(commercial)*100:.1f}%)")

if len(com_with_costs) > 0:
    print(f"\n  Buildings with upgrades - Cost distribution:")
    print(f"    Mean: ${com_with_costs['weighted_cost'].mean():,.0f}")
    print(f"    Median: ${com_with_costs['weighted_cost'].median():,.0f}")
    print(f"    Range: ${com_with_costs['weighted_cost'].min():,.0f} - ${com_with_costs['weighted_cost'].max():,.0f}")


COST VARIATION WITHIN BUILDING TYPES

Residential Buildings:
  With upgrades: 120,923 (22.0%)
  Without upgrades: 0 (0.0%)

  Buildings with upgrades - Cost distribution:
    Mean: $65,130
    Median: $37,510
    Range: $234 - $507,416

Commercial Buildings:
  With upgrades: 51,882 (15.5%)
  Without upgrades: 0 (0.0%)

  Buildings with upgrades - Cost distribution:
    Mean: $15,709
    Median: $9,084
    Range: $951 - $198,517


## Cost Uncertainty Ranges

Let's explore the range of possible total costs due to distribution variability.

In [23]:
# Analyze which buildings drive costs
buildings_with_upgrades = results[results['weighted_cost'] > 0]

print("\n" + "="*70)
print("BUILDINGS DRIVING UPGRADE COSTS")
print("="*70)

print(f"\nTotal buildings analyzed: {len(results):,}")
print(f"Buildings requiring upgrades: {len(buildings_with_upgrades):,} ({len(buildings_with_upgrades)/len(results)*100:.1f}%)")
print(f"\nTotal upgrade costs: ${results['weighted_cost'].sum():,.0f}")
print(f"Cost per building (all): ${results['weighted_cost'].mean():,.0f}")
print(f"Cost per building (with upgrades): ${buildings_with_upgrades['weighted_cost'].mean():,.0f}")

# Top cost buildings
top_10_pct = results.nlargest(int(len(results) * 0.10), 'weighted_cost')
top_10_pct_cost = top_10_pct['weighted_cost'].sum()
print(f"\nTop 10% of buildings:")
print(f"  Count: {len(top_10_pct):,}")
print(f"  Total cost: ${top_10_pct_cost:,.0f}")
print(f"  Percentage of total: {top_10_pct_cost/results['weighted_cost'].sum()*100:.1f}%")


BUILDINGS DRIVING UPGRADE COSTS

Total buildings analyzed: 884,596
Buildings requiring upgrades: 172,805 (19.5%)

Total upgrade costs: $7,573,495,293
Cost per building (all): $43,821
Cost per building (with upgrades): $50,292

Top 10% of buildings:
  Count: 88,459
  Total cost: $7,801,113,297
  Percentage of total: 103.0%


## Understanding Cost Variation

Let's analyze how costs vary across buildings and what factors drive variation.

In [16]:
# Analyze state-level cost variation
print("\n" + "="*70)
print("STATE-LEVEL COST VARIATION")
print("="*70)

state_costs = results.groupby('state')['weighted_cost'].agg(['count', 'sum', 'mean', 'std', 'min', 'max'])
state_costs = state_costs.sort_values('sum', ascending=False)

print("\nTop 10 States by Total Cost:")
print(f"{'State':<5} {'Buildings':>12} {'Total Cost':>20} {'Mean Cost':>15} {'Std Dev':>15}")
print("-" * 70)
for state, row in state_costs.head(10).iterrows():
    print(f"{state:<5} {row['count']:>12,.0f} ${row['sum']:>18,.0f} ${row['mean']:>13,.0f} ${row['std']:>13,.0f}")


STATE-LEVEL COST VARIATION

Top 10 States by Total Cost:
State    Buildings           Total Cost       Mean Cost         Std Dev
----------------------------------------------------------------------
CA          19,336 $     1,096,940,988 $       56,731 $       76,091
TX          11,933 $       509,608,501 $       42,706 $       60,342
MI          10,196 $       498,461,227 $       48,888 $       65,362
NY           6,932 $       422,933,747 $       61,012 $       78,707
IL           6,848 $       378,266,081 $       55,237 $       73,057
FL           6,994 $       349,711,313 $       50,002 $       69,658
OH           7,024 $       325,192,398 $       46,297 $       63,362
PA           5,970 $       314,227,774 $       52,634 $       69,221
NJ           5,086 $       282,970,833 $       55,637 $       74,717
NC           5,873 $       259,295,244 $       44,150 $       60,187


In [17]:
# Analyze cost variation patterns
print("\n" + "="*70)
print("COST VARIATION PATTERNS")
print("="*70)

# Distribution shape analysis
print("\nDistribution characteristics:")
print(f"  Skewness: {stats.skew(results['weighted_cost']):.2f}")
print(f"    (Positive = right-skewed: many low costs, few high costs)")

print(f"\n  Kurtosis: {stats.kurtosis(results['weighted_cost']):.2f}")
print(f"    (Measures tail heaviness)")

# Compare distribution shapes by state
print("\nCost variation by state (top 5 states):")
for state in state_costs.head(5).index:
    state_data = results[results['state'] == state]['weighted_cost']
    print(f"\n  {state}:")
    print(f"    Mean: ${state_data.mean():,.0f}")
    print(f"    Median: ${state_data.median():,.0f}")
    print(f"    Skewness: {stats.skew(state_data):.2f}")
    print(f"    Coefficient of variation: {state_data.std()/state_data.mean():.2f}")


COST VARIATION PATTERNS

Distribution characteristics:
  Skewness: nan
    (Positive = right-skewed: many low costs, few high costs)

  Kurtosis: nan
    (Measures tail heaviness)

Cost variation by state (top 5 states):

  CA:
    Mean: $56,731
    Median: $28,234
    Skewness: nan
    Coefficient of variation: 1.34

  TX:
    Mean: $42,706
    Median: $19,448
    Skewness: nan
    Coefficient of variation: 1.41

  MI:
    Mean: $48,888
    Median: $23,620
    Skewness: nan
    Coefficient of variation: 1.34

  NY:
    Mean: $61,012
    Median: $31,037
    Skewness: nan
    Coefficient of variation: 1.29

  IL:
    Mean: $55,237
    Median: $27,416
    Skewness: nan
    Coefficient of variation: 1.32


## Cost Drivers and Distributions

In [24]:
# Analyze cost drivers by examining upgrade requirements
print("\n" + "="*70)
print("COST DRIVERS: UPGRADE REQUIREMENTS BY TECHNOLOGY")
print("="*70)

# Buildings with different capacity requirements
buildings_with_costs = results[results['weighted_cost'] > 0]

# Analyze required capacities that drive upgrades
print(f"\nAmong {len(buildings_with_costs):,} buildings needing upgrades:")

# Check which have positive max_elec_consumption_kwh
high_consumption = buildings_with_costs[buildings_with_costs['max_elec_consumption_kwh'] > 0]
print(f"  High electrical consumption: {len(high_consumption):,} ({len(high_consumption)/len(buildings_with_costs)*100:.1f}%)")

# Average consumption for buildings with/without costs
avg_consumption_all = results['max_elec_consumption_kwh'].mean()
avg_consumption_upgraded = buildings_with_costs['max_elec_consumption_kwh'].mean()

print(f"\nAverage Max Electrical Consumption:")
print(f"  All buildings: {avg_consumption_all:,.0f} kWh/year")
print(f"  Buildings needing upgrades: {avg_consumption_upgraded:,.0f} kWh/year")

# Cost relationship with building size
print(f"\nCost Correlation with Building Size (sqft):")
correlation = results['sqft'].corr(results['weighted_cost'])
print(f"  Correlation coefficient: {correlation:.3f}")
print(f"  (Positive = larger buildings tend to have higher costs)")


COST DRIVERS: UPGRADE REQUIREMENTS BY TECHNOLOGY

Among 172,805 buildings needing upgrades:
  High electrical consumption: 172,805 (100.0%)

Average Max Electrical Consumption:
  All buildings: 23 kWh/year
  Buildings needing upgrades: 15 kWh/year

Cost Correlation with Building Size (sqft):
  Correlation coefficient: -0.012
  (Positive = larger buildings tend to have higher costs)


## Designing Custom Distributions

If you need to use custom cost distributions, consider these approaches.

In [25]:
# Example: Creating a custom cost distribution

print("\n" + "="*70)
print("CREATING CUSTOM COST DISTRIBUTIONS")
print("="*70)

# Approach 1: Empirical Distribution (from your data)
def create_empirical_distribution(cost_data):
    """
    Create an empirical distribution from observed cost data.
    
    Args:
        cost_data: array of observed costs (non-zero values)
    
    Returns:
        A function that samples from the empirical distribution
    """
    # Filter to non-zero costs to match BICEP approach
    non_zero_costs = cost_data[cost_data > 0].values
    sorted_costs = np.sort(non_zero_costs)
    
    def sample(n=1):
        return np.random.choice(sorted_costs, size=n, replace=True)
    
    return sample, sorted_costs

# Create empirical distribution from actual residential data
res_empirical, res_costs_sorted = create_empirical_distribution(residential['weighted_cost'])

print(f"\nApproach 1: Empirical Distribution")
print(f"  Created from {len(res_costs_sorted):,} buildings with costs")
print(f"  Sample from distribution (10 buildings):")
samples = res_empirical(10)
for i, cost in enumerate(samples, 1):
    print(f"    Building {i}: ${cost:,.0f}")

# Approach 2: Parametric Distribution (log-normal)
def create_lognormal_distribution(data, min_val=0, max_val=np.inf):
    """
    Create a log-normal distribution fitted to cost data.
    
    Log-normal distributions are common for costs (always positive, right-skewed).
    
    Args:
        data: array of observed costs (non-zero)
        min_val: minimum cost (constrain)
        max_val: maximum cost (constrain)
    
    Returns:
        A function that samples from the distribution and fit parameters
    """
    non_zero = data[data > 0]
    # Fit log-normal to the data
    params = stats.lognorm.fit(non_zero)
    
    def sample(n=1):
        samples = np.random.lognormal(params[1], params[0], size=n)
        samples = np.clip(samples, min_val, max_val)
        return samples
    
    return sample, params

# Create log-normal distribution from actual residential data
res_lognormal, params = create_lognormal_distribution(residential['weighted_cost'])

print(f"\nApproach 2: Log-Normal Distribution")
print(f"  Fitted to actual residential data")
print(f"  Sample from distribution (10 buildings):")
samples = res_lognormal(10)
for i, cost in enumerate(samples, 1):
    print(f"    Building {i}: ${cost:,.0f}")


CREATING CUSTOM COST DISTRIBUTIONS

Approach 1: Empirical Distribution
  Created from 120,923 buildings with costs
  Sample from distribution (10 buildings):
    Building 1: $215,640
    Building 2: $94,520
    Building 3: $15,075
    Building 4: $451,657
    Building 5: $15,598
    Building 6: $72,592
    Building 7: $56,841
    Building 8: $14,200
    Building 9: $182,292
    Building 10: $20,610

Approach 2: Log-Normal Distribution
  Fitted to actual residential data
  Sample from distribution (10 buildings):
    Building 1: $0
    Building 2: $0
    Building 3: $0
    Building 4: $0
    Building 5: $0
    Building 6: $0
    Building 7: $0
    Building 8: $0
    Building 9: $0
    Building 10: $0


In [26]:
# Compare default vs custom distributions

print("\n" + "="*70)
print("COMPARING DISTRIBUTION APPROACHES")
print("="*70)

# Generate samples from both approaches
empirical_samples = res_empirical(5000)
lognormal_samples = res_lognormal(5000)
actual_data = residential[residential['weighted_cost'] > 0]['weighted_cost'].values

print(f"\nStatistics Comparison:")
print(f"{'Metric':<30} {'Actual Data':>20} {'Empirical':>20} {'Log-Normal':>20}")
print("-" * 72)
print(f"{'Mean':<30} ${np.mean(actual_data):>18,.0f} ${np.mean(empirical_samples):>18,.0f} ${np.mean(lognormal_samples):>18,.0f}")
print(f"{'Median':<30} ${np.median(actual_data):>18,.0f} ${np.median(empirical_samples):>18,.0f} ${np.median(lognormal_samples):>18,.0f}")
print(f"{'Std Dev':<30} ${np.std(actual_data):>18,.0f} ${np.std(empirical_samples):>18,.0f} ${np.std(lognormal_samples):>18,.0f}")
print(f"{'Min':<30} ${np.min(actual_data):>18,.0f} ${np.min(empirical_samples):>18,.0f} ${np.min(lognormal_samples):>18,.0f}")
print(f"{'Max':<30} ${np.max(actual_data):>18,.0f} ${np.max(empirical_samples):>18,.0f} ${np.max(lognormal_samples):>18,.0f}")


COMPARING DISTRIBUTION APPROACHES

Statistics Comparison:
Metric                                  Actual Data            Empirical           Log-Normal
------------------------------------------------------------------------
Mean                           $            65,130 $            62,770 $                 0
Median                         $            37,510 $            37,454 $                 0
Std Dev                        $            75,607 $            71,699 $                 0
Min                            $               234 $               724 $                 0
Max                            $           507,416 $           473,477 $                 0


## Impact of Distribution Choice on Total Costs

Different distributions can lead to significantly different total cost estimates.

In [27]:
# Impact of distribution choice on total costs

print("\n" + "="*70)
print("IMPACT OF DISTRIBUTION CHOICE ON TOTAL COSTS")
print("="*70)

# Get the count of residential buildings needing upgrades
num_buildings_upgraded = len(res_with_costs)

# Calculate total costs with different distributions
actual_mean = actual_data.mean()
empirical_mean = empirical_samples.mean()
lognormal_mean = lognormal_samples.mean()

total_actual = actual_mean * num_buildings_upgraded
total_empirical = empirical_mean * num_buildings_upgraded
total_lognormal = lognormal_mean * num_buildings_upgraded

print(f"\nProjecting costs for {num_buildings_upgraded:,} residential buildings needing upgrades:")
print(f"\nUsing Actual Data Distribution:")
print(f"  Mean cost per building: ${actual_mean:,.0f}")
print(f"  Total estimated cost: ${total_actual:,.0f}")

print(f"\nUsing Empirical Distribution (resampling actual):")
print(f"  Mean cost per building: ${empirical_mean:,.0f}")
print(f"  Total estimated cost: ${total_empirical:,.0f}")

print(f"\nUsing Log-Normal Distribution (parametric fit):")
print(f"  Mean cost per building: ${lognormal_mean:,.0f}")
print(f"  Total estimated cost: ${total_lognormal:,.0f}")

# Calculate differences
empirical_diff = total_empirical - total_actual
lognormal_diff = total_lognormal - total_actual
empirical_pct = (empirical_diff / total_actual) * 100
lognormal_pct = (lognormal_diff / total_actual) * 100

print(f"\nDifferences from actual data:")
print(f"  Empirical: ${empirical_diff:+,.0f} ({empirical_pct:+.1f}%)")
print(f"  Log-Normal: ${lognormal_diff:+,.0f} ({lognormal_pct:+.1f}%)")


IMPACT OF DISTRIBUTION CHOICE ON TOTAL COSTS

Projecting costs for 120,923 residential buildings needing upgrades:

Using Actual Data Distribution:
  Mean cost per building: $65,130
  Total estimated cost: $7,875,737,260

Using Empirical Distribution (resampling actual):
  Mean cost per building: $62,770
  Total estimated cost: $7,590,312,778

Using Log-Normal Distribution (parametric fit):
  Mean cost per building: $0
  Total estimated cost: $0

Differences from actual data:
  Empirical: $-285,424,482 (-3.6%)
  Log-Normal: $-7,875,737,260 (-100.0%)


## Key Takeaways

1. **Distributions capture uncertainty**: Costs aren't fixed; they vary based on many factors
2. **Distribution choice matters**: Different assumptions lead to significantly different estimates
3. **BICEP uses empirical and parametric distributions**: Based on real cost data and expert judgment
4. **Customization is possible**: You can implement your own distributions if needed
5. **Sensitivity analysis is important**: Test how results change with different cost assumptions

## Next Steps

- Review [Data Requirements](data-requirements.md) to understand technology adoption patterns
- Check [Scenario Comparison](scenario-comparison.md) for cost implications across scenarios
- Consult the [API Reference](../api-reference.md) for distribution class documentation